# Übung - Multiple Lineare Regression

Es ist an der Zeit, Ihr Wissen auf einen neuen Datensatz anzuwenden. Damit nicht zu viele Änderungen auf einmal kommen, bleiben wir beim Auto-Thema und arbeiten mit den **Autositze-Daten**. Ein Unternehmen, das Autositze herstellt, möchte ein Modell erstellen, um Verkäufe vorherzusagen, und sie brauchen Ihre Hilfe!

Sie finden die Datei `carseats.csv` im data-Ordner. Sie enthält 400 Beobachtungen zu den folgenden 11 Variablen:
* **Sales**: Einheitsverkäufe (in Tausend) an jedem Standort
* **CompPrice**: Preis, den der Wettbewerber an jedem Standort verlangt
* **Income**: Gemeinschaftseinkommensniveau (in Tausend Dollar)
* **Advertising**: Lokales Werbebudget des Unternehmens an jedem Standort (in Tausend Dollar)
* **Population**: Bevölkerungsgröße in der Region (in Tausend)
* **Price**: Preis, den das Unternehmen für Autositze an jedem Standort verlangt
* **ShelveLoc**: Ein Faktor mit den Stufen Bad, Good und Medium, der die Qualität der Regalplatzierung für die Autositze an jedem Standort angibt
* **Age**: Durchschnittsalter der lokalen Bevölkerung
* **Education**: Bildungsniveau an jedem Standort
* **Urban**: Ein Faktor mit den Stufen No und Yes, um anzuzeigen, ob das Geschäft an einem städtischen oder ländlichen Standort liegt
* **US**: Ein Faktor mit den Stufen No und Yes, um anzuzeigen, ob das Geschäft in den USA liegt oder nicht

**(a) Laden Sie die Daten**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from itertools import combinations
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

sns.set_style("ticks")
plt.rcParams["figure.figsize"] = (7, 4)

carseats = pd.read_csv("data/carseats.csv")
carseats_encoded = pd.get_dummies(carseats, drop_first=True)


def adjusted_r_squared(r_squared, X):
    return 1 - ((1 - r_squared) * (len(X) - 1) / (len(X) - X.shape[1] - 1))


def fit_linear_model(feature_names):
    X = carseats_encoded[list(feature_names)]
    y = carseats_encoded["Sales"]

    model = LinearRegression()
    model.fit(X, y)

    predictions = model.predict(X)
    r_squared = r2_score(y, predictions)
    adjusted_r2 = adjusted_r_squared(r_squared, X)

    return model, X, y, predictions, r_squared, adjusted_r2


feature_names = [column for column in carseats_encoded.columns if column != "Sales"]

carseats.head()

**(b) Visualisieren Sie die Daten mit den entsprechenden Plots.**

<br>
<details><summary>
Hier klicken für einen Hinweis…
</summary>
Schauen Sie sich die Dokumentation für seaborns pair plots an.
</details>

In [ ]:
numeric_columns = [
    "Sales",
    "CompPrice",
    "Income",
    "Advertising",
    "Population",
    "Price",
    "Age",
    "Education",
]

sns.pairplot(carseats[numeric_columns], corner=True);

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=carseats, x="ShelveLoc", y="Sales", ax=axes[0])
sns.boxplot(data=carseats, x="Urban", y="Sales", ax=axes[1])
sns.boxplot(data=carseats, x="US", y="Sales", ax=axes[2])

axes[0].set_title("Sales by ShelveLoc")
axes[1].set_title("Sales by Urban")
axes[2].set_title("Sales by US");

**(c) Welche Trends sehen Sie in den Daten?**

In [ ]:
"""
- Hoehere Preise gehen tendenziell mit niedrigeren Sales einher.
- Mehr Advertising haengt oft mit hoeheren Sales zusammen.
- ShelveLoc ist sehr auffaellig: Good liefert deutlich hoehere Sales als Medium oder Bad.
- Urban wirkt schwach, waehrend US hoechstens einen kleineren Effekt zu haben scheint.
"""

**(d) Finden Sie den einzelnen besten Prädiktor für eine einfache lineare Regression.**

<br>
<details><summary>
Hier klicken für einen Hinweis…
</summary>
Passen Sie ein lineares Modell an alle möglichen erklärenden Variablen an und wählen Sie das beste aus.
</details>

In [ ]:
single_feature_results = []

for feature in feature_names:
    _, X_single, _, _, r_squared, adjusted_r2 = fit_linear_model([feature])
    single_feature_results.append(
        {
            "feature": feature,
            "r_squared": r_squared,
            "adjusted_r_squared": adjusted_r2,
        }
    )

single_feature_results = (
    pd.DataFrame(single_feature_results)
    .sort_values("r_squared", ascending=False)
    .reset_index(drop=True)
)

print("Best single predictor:", single_feature_results.loc[0, "feature"])
single_feature_results.head()

**(e) Passen Sie ein Modell mit allen möglichen erklärenden Variablen an.**

In [ ]:
all_features_model, X_all, y_all, y_hat_all, all_r_squared, all_adjusted_r2 = fit_linear_model(feature_names)

coefficient_table = pd.DataFrame(
    {
        "feature": X_all.columns,
        "coefficient": all_features_model.coef_,
    }
)

print("Intercept:", round(all_features_model.intercept_, 3))
print("R-squared:", round(all_r_squared, 3))
print("Adjusted R-squared:", round(all_adjusted_r2, 3))

coefficient_table

**(f) Was ist das beste Modell laut $R^2$?**

In [ ]:
subset_results = []

for n_features in range(1, len(feature_names) + 1):
    for subset in combinations(feature_names, n_features):
        _, X_subset, _, _, r_squared, adjusted_r2 = fit_linear_model(subset)
        subset_results.append(
            {
                "n_features": len(subset),
                "features": subset,
                "r_squared": r_squared,
                "adjusted_r_squared": adjusted_r2,
            }
        )

subset_results = (
    pd.DataFrame(subset_results)
    .sort_values(["r_squared", "adjusted_r_squared"], ascending=False)
    .reset_index(drop=True)
)

best_r2_model = subset_results.iloc[0]
print("Best model according to R-squared:", best_r2_model["features"])
print("R-squared:", round(best_r2_model["r_squared"], 3))

subset_results.head()

**(g) Entfernen Sie einige erklärende Variablen. Wie ändert sich $R^2$?**

In [ ]:
reduced_models = {
    "all_features": feature_names,
    "without_Urban_Yes": [feature for feature in feature_names if feature != "Urban_Yes"],
    "without_Urban_Yes_and_Population": [
        feature for feature in feature_names if feature not in ["Urban_Yes", "Population"]
    ],
    "without_Urban_Yes_Population_US_Yes": [
        feature for feature in feature_names
        if feature not in ["Urban_Yes", "Population", "US_Yes"]
    ],
}

reduced_model_results = []

for name, subset in reduced_models.items():
    _, _, _, _, r_squared, adjusted_r2 = fit_linear_model(subset)
    reduced_model_results.append(
        {
            "model": name,
            "n_features": len(subset),
            "r_squared": r_squared,
            "adjusted_r_squared": adjusted_r2,
        }
    )

pd.DataFrame(reduced_model_results).sort_values("r_squared", ascending=False)

**(h) Wiederholen Sie den Prozess für das adjustierte $R^2$.**

In [ ]:
best_adjusted_r2_model = subset_results.sort_values(
    "adjusted_r_squared", ascending=False
).iloc[0]

print(
    "Best model according to adjusted R-squared:",
    best_adjusted_r2_model["features"],
)
print(
    "Adjusted R-squared:",
    round(best_adjusted_r2_model["adjusted_r_squared"], 3),
)

subset_results.sort_values("adjusted_r_squared", ascending=False).head()

**(i) Was sind Ihre interessantesten Erkenntnisse?**

In [ ]:
"""
- ShelveLoc_Good ist der staerkste einzelne Praediktor und schlaegt jede einzelne numerische Variable.
- Das volle Modell liefert das hoechste normale R-squared, aber das adjustierte R-squared bevorzugt ein etwas kleineres Modell.
- Population und Urban_Yes tragen kaum etwas bei; ihr Entfernen kostet fast keine Modellguete.
- Price wirkt klar negativ auf Sales, waehrend Advertising und eine gute Regalplatzierung positiv wirken.
"""

<br>
<br>
<br>

----